#Extension Task - 1

In [ ]:

%%capture
!pip install langchain langchain-core langchain-groq langchain-openai rouge-score

In [ ]:
from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate
)

from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

from rouge_score import rouge_scorer

In [ ]:
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_KEY"],
    temperature=0.3
)

##Task 1: Earnings Call Summarization

Zero-Shot Prompt

In [ ]:
import langchain
print(langchain.__version__)

1.3.4


In [ ]:
earnings_generation_prompt = PromptTemplate(
    input_variables=["company", "quarter"],
    template="""
You are a financial analyst.

Generate a realistic earnings call transcript snippet for {company} during {quarter}.

Include:
- Revenue discussion
- Profitability discussion
- Future guidance
- Management commentary

Length: 150-200 words.
"""
)

generation_chain = (
    earnings_generation_prompt
    | llm
    | StrOutputParser()
)

snippet = generation_chain.invoke(
    {
        "company": "Microsoft",
        "quarter": "Q2 FY2025"
    }
)

print(snippet)

"Good morning, everyone. I'm Amy Hood, CFO of Microsoft. For Q2 FY2025, we reported revenue of $56.2 billion, up 10% year-over-year. Our cloud segment drove growth, with Azure revenue increasing 27%. 

Our operating income was $22.8 billion, representing a 12% increase. We saw strong profitability in our Productivity and Business Processes segment, with operating margins expanding to 43%. 

Looking ahead, we expect Q3 revenue to be between $54.5 and $56.5 billion. We're confident in our ability to drive long-term growth, with a focus on cloud, AI, and hybrid work solutions. 

As Satya mentioned, our investments in AI are paying off, with significant customer adoption of our Azure AI platform. We're well-positioned to capitalize on emerging trends and continue to deliver value to our shareholders."


Few-Shot Earnings Call Generation

In [ ]:
examples = [
    {
        "company": "Apple",
        "quarter": "Q1 FY2025",
        "snippet": """
Revenue increased 8% year-over-year driven by strong iPhone and Services growth.
Operating margins improved due to cost optimization.
Management expects continued demand in the upcoming quarter.
"""
    },
    {
        "company": "Amazon",
        "quarter": "Q4 FY2024",
        "snippet": """
AWS continued double-digit growth while advertising revenue exceeded expectations.
Profitability improved through operational efficiencies.
Management provided positive guidance for the next quarter.
"""
    }
]
example_prompt = PromptTemplate.from_template(
"""
Company: {company}

Quarter: {quarter}

Transcript:
{snippet}
"""
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
You are a financial analyst.

Generate realistic earnings call snippets following the style of the examples.
""",
    suffix="""
Company: {company}

Quarter: {quarter}

Transcript:
""",
    input_variables=["company", "quarter"]
)
few_shot_chain = (
    few_shot_prompt
    | llm
    | StrOutputParser()
)
result = few_shot_chain.invoke(
    {
        "company": "NVIDIA",
        "quarter": "Q1 FY2026"
    }
)

print(result)

Company: NVIDIA

Quarter: Q1 FY2026

Transcript:

Revenue grew 12% year-over-year, driven by exceptional demand for our GeForce and datacenter products.
Gross margins expanded due to a favorable mix of high-end graphics cards and increasing adoption of our AI computing solutions.
Management expects sustained momentum in the fields of artificial intelligence, gaming, and cloud computing, and is confident in achieving strong growth throughout the fiscal year.


Zero-Shot Summarization

In [ ]:
summary_prompt = PromptTemplate.from_template(
"""
Summarize the following earnings call transcript.

Include:

1. Revenue Performance
2. Profitability
3. Key Drivers
4. Future Outlook

Transcript:

{transcript}

Summary:
"""
)
summary_chain = (
    summary_prompt
    | llm
    | StrOutputParser()
)
summary = summary_chain.invoke(
    {
        "transcript": snippet
    }
)

print(summary)

Here is a summary of the earnings call transcript:

1. **Revenue Performance**: Microsoft reported revenue of $56.2 billion for Q2 FY2025, representing a 10% year-over-year increase, driven by strong growth in the cloud segment, particularly Azure, which saw a 27% revenue increase.

2. **Profitability**: The company's operating income was $22.8 billion, a 12% increase, with strong profitability in the Productivity and Business Processes segment, which had operating margins of 43%.

3. **Key Drivers**: The key drivers of Microsoft's growth were its cloud segment, particularly Azure, as well as its investments in AI, which are paying off with significant customer adoption of the Azure AI platform.

4. **Future Outlook**: Looking ahead, Microsoft expects Q3 revenue to be between $54.5 and $56.5 billion and is confident in its ability to drive long-term growth, with a focus on cloud, AI, and hybrid work solutions, positioning the company to capitalize on emerging trends and deliver value t

Few-Shot Summarization

In [ ]:
summary_examples = [
    {
        "transcript": """
Revenue grew 15% due to cloud services.
Margins expanded because of cost efficiencies.
Management expects strong growth next quarter.
""",
        "summary": """
Revenue increased 15% driven by cloud growth.
Margins improved through operational efficiencies.
Management provided positive guidance.
"""
    }
]
summary_example_prompt = PromptTemplate.from_template(
"""
Transcript:
{transcript}

Summary:
{summary}
"""
)
few_shot_summary_prompt = FewShotPromptTemplate(
    examples=summary_examples,
    example_prompt=summary_example_prompt,
    prefix="""
Summarize earnings call transcripts.
""",
    suffix="""
Transcript:
{transcript}

Summary:
""",
    input_variables=["transcript"]
)
few_shot_summary_chain = (
    few_shot_summary_prompt
    | llm
    | StrOutputParser()
)
summary = few_shot_summary_chain.invoke(
    {
        "transcript": snippet
    }
)

print(summary)

Microsoft's Q2 FY2025 revenue was $56.2 billion, up 10% year-over-year, driven by cloud growth with Azure revenue increasing 27%. Operating income rose 12% to $22.8 billion, with expanding margins in the Productivity and Business Processes segment. The company expects Q3 revenue to be between $54.5 and $56.5 billion and is confident in its ability to drive long-term growth, fueled by investments in cloud, AI, and hybrid work solutions.


ROUGE-L Evaluation

In [ ]:
reference_summary = """
Revenue increased due to cloud adoption.
Margins improved through cost efficiencies.
Management expects continued growth.
"""

generated_summary = summary
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

scores = scorer.score(
    reference_summary,
    generated_summary
)

print(scores["rougeL"])

Score(precision=0.07894736842105263, recall=0.4, fmeasure=0.13186813186813187)


Generate 3 Earnings Call Snippets + Summaries + ROUGE-L Together

In [ ]:
companies = [
    ("Microsoft", "Q2 FY2025"),
    ("NVIDIA", "Q1 FY2026"),
    ("Amazon", "Q4 FY2025")
]

for company, quarter in companies:

    transcript = generation_chain.invoke(
        {
            "company": company,
            "quarter": quarter
        }
    )

    summary = summary_chain.invoke(
        {
            "transcript": transcript
        }
    )

    print("=" * 80)
    print(company)
    print("=" * 80)

    print("\nTRANSCRIPT\n")
    print(transcript)

    print("\nSUMMARY\n")
    print(summary)

Microsoft

TRANSCRIPT

Amy Hood, CFO: "For Q2 FY2025, Microsoft reported revenue of $56.2 billion, up 10% year-over-year. Our cloud segment drove growth, with Azure revenue increasing 27%. 

Operating income was $22.8 billion, representing a 20% margin. We saw improved profitability in our Productivity and Business Processes segment, driven by Office 365 and LinkedIn.

Looking ahead, we expect Q3 revenue to be between $54.5 and $55.5 billion. We anticipate continued cloud growth, with Azure revenue expected to increase over 25% year-over-year.

Satya Nadella, CEO: "Our results demonstrate the strength of our cloud-first strategy. We're seeing significant traction with our Azure AI and machine learning offerings, and our Microsoft 365 suite continues to resonate with customers. We're well-positioned for long-term growth and remain committed to investing in innovation and customer success."

SUMMARY

Here's a summary of the earnings call transcript:

1. **Revenue Performance**: Microsoft